# 15 — System wsparcia decyzji operatora farmy PV

## Cel

Ten notebook zamienia wyniki backtestu w końcową warstwę projektu:

1. porównuje trzy scenariusze na podstawie wyniku po korekcie końcowego SOC,
2. wskazuje zwycięską strategię,
3. tworzy jedną wykonalną rekomendację operatora na każdy interwał 15-minutowy,
4. zapisuje ranking, KPI oraz szczegółowy replay decyzji w warstwie Gold.

Notebook **nie trenuje ponownie modeli**. Korzysta wyłącznie z zatwierdzonych
wyników notebooków 13–14.

Ważne: wybór zwycięzcy opisuje wynik historycznego backtestu. W zastosowaniu
produkcyjnym strategię należy zamrozić, a następnie ocenić na kolejnym,
niewidzianym okresie.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "workspace"
GOLD_SCHEMA = "energy_gold"

BACKTEST_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.battery_dispatch_backtest_15m"
)
SUMMARY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.battery_backtest_summary"
)
RANKING_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.decision_support_strategy_ranking"
)
KPI_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.decision_support_kpi"
)
REPLAY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.daily_decision_replay_15m"
)

EXPECTED_SCENARIOS = [
    "NO_BATTERY",
    "BATTERY_BALANCING_PV_ONLY",
    "BATTERY_PRICE_GRID",
]

for table_name in [BACKTEST_TABLE, SUMMARY_TABLE]:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Brak wymaganej tabeli: {table_name}")

backtest_df = spark.table(BACKTEST_TABLE)
summary_df = spark.table(SUMMARY_TABLE)

print(f"Backtest: {backtest_df.count()} rekordów")
print(f"Podsumowanie: {summary_df.count()} rekordów")


## 1. Ranking strategii

Podstawowym kryterium jest `adjusted_net_revenue_usd`, czyli wynik obejmujący:

- rozliczenie DAM i FMM,
- koszt degradacji baterii,
- korektę wartości końcowego SOC.

Przy identycznym wyniku preferowana jest strategia z mniejszym odchyleniem od
harmonogramu, a następnie z mniejszą liczbą cykli.

In [0]:
ranking_window = Window.orderBy(
    F.desc("adjusted_net_revenue_usd"),
    F.asc("absolute_schedule_deviation_mwh"),
    F.asc("equivalent_cycles"),
)

scenario_labels = F.create_map(
    F.lit("NO_BATTERY"),
    F.lit("Brak magazynu — scenariusz bazowy"),
    F.lit("BATTERY_BALANCING_PV_ONLY"),
    F.lit("Bilansowanie harmonogramu energią z PV"),
    F.lit("BATTERY_PRICE_GRID"),
    F.lit("Sterowanie cenowe z możliwością ładowania z sieci"),
)

ranking_df = (
    summary_df
    .withColumn(
        "strategy_rank",
        F.row_number().over(ranking_window),
    )
    .withColumn(
        "is_recommended_strategy",
        F.col("strategy_rank") == F.lit(1),
    )
    .withColumn(
        "strategy_description",
        scenario_labels[F.col("scenario")],
    )
    .withColumn(
        "recommendation_status",
        F.when(
            F.col("strategy_rank") == 1,
            F.lit("RECOMMENDED"),
        ).otherwise(F.lit("ALTERNATIVE")),
    )
    .withColumn(
        "selection_criterion",
        F.lit(
            "MAX_ADJUSTED_NET_REVENUE_THEN_MIN_DEVIATION_AND_CYCLES"
        ),
    )
    .withColumn(
        "recommendation_scope",
        F.lit("HISTORICAL_BACKTEST_SELECTION"),
    )
    .withColumn("_created_at_utc", F.current_timestamp())
)

(
    ranking_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RANKING_TABLE)
)

winner_rows = (
    ranking_df
    .filter(F.col("is_recommended_strategy"))
    .collect()
)

if len(winner_rows) != 1:
    raise RuntimeError("Ranking nie wskazał dokładnie jednej strategii.")

winner = winner_rows[0].asDict()
RECOMMENDED_STRATEGY = winner["scenario"]

print(f"Rekomendowana strategia: {RECOMMENDED_STRATEGY}")
display(
    ranking_df
    .orderBy("strategy_rank")
    .select(
        "strategy_rank",
        "scenario",
        "recommendation_status",
        "adjusted_net_revenue_usd",
        "revenue_difference_vs_no_battery_usd",
        "absolute_schedule_deviation_mwh",
        "equivalent_cycles",
        "total_grid_import_mwh",
        "final_soc_mwh",
    )
)


## 2. KPI końcowej rekomendacji

Jednowierszowa tabela KPI odpowiada na pytania biznesowe:

- którą strategię rekomenduje system,
- jaki wynik daje względem braku baterii,
- jak wpływa na odchylenia od harmonogramu,
- ile wykorzystuje cykli i energii z sieci,
- czy magazyn tworzy dodatnią wartość po pełnych kosztach.

In [0]:
winner_df = (
    ranking_df
    .filter(F.col("is_recommended_strategy"))
    .alias("winner")
)

baseline_df = (
    ranking_df
    .filter(F.col("scenario") == "NO_BATTERY")
    .alias("baseline")
)

kpi_df = (
    winner_df
    .crossJoin(baseline_df)
    .select(
        F.col("winner.scenario").alias("recommended_strategy"),
        F.col("winner.strategy_description").alias(
            "recommended_strategy_description"
        ),
        F.col("winner.adjusted_net_revenue_usd").alias(
            "recommended_adjusted_net_revenue_usd"
        ),
        F.col("baseline.adjusted_net_revenue_usd").alias(
            "no_battery_adjusted_net_revenue_usd"
        ),
        (
            F.col("winner.adjusted_net_revenue_usd")
            - F.col("baseline.adjusted_net_revenue_usd")
        ).alias("incremental_revenue_usd"),
        (
            F.lit(100.0)
            * (
                F.col("winner.adjusted_net_revenue_usd")
                - F.col("baseline.adjusted_net_revenue_usd")
            )
            / F.col("baseline.adjusted_net_revenue_usd")
        ).alias("incremental_revenue_percent"),
        F.col("winner.absolute_schedule_deviation_mwh").alias(
            "recommended_schedule_deviation_mwh"
        ),
        F.col("baseline.absolute_schedule_deviation_mwh").alias(
            "no_battery_schedule_deviation_mwh"
        ),
        (
            F.lit(100.0)
            * (
                F.col("baseline.absolute_schedule_deviation_mwh")
                - F.col("winner.absolute_schedule_deviation_mwh")
            )
            / F.col("baseline.absolute_schedule_deviation_mwh")
        ).alias("schedule_deviation_reduction_percent"),
        F.col("winner.equivalent_cycles").alias("equivalent_cycles"),
        F.col("winner.total_grid_import_mwh").alias(
            "total_grid_import_mwh"
        ),
        F.col("winner.final_soc_mwh").alias("final_soc_mwh"),
        F.when(
            (F.col("winner.scenario") != "NO_BATTERY")
            & (
                F.col("winner.adjusted_net_revenue_usd")
                > F.col("baseline.adjusted_net_revenue_usd")
            ),
            F.lit("RECOMMEND_BATTERY_STRATEGY"),
        ).otherwise(F.lit("NO_BATTERY_ADVANTAGE_CONFIRMED"))
        .alias("business_recommendation"),
        F.lit("HISTORICAL_BACKTEST_SELECTION").alias(
            "recommendation_scope"
        ),
        F.current_timestamp().alias("_created_at_utc"),
    )
)

(
    kpi_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(KPI_TABLE)
)

display(kpi_df)


## 3. Jedna decyzja operatora na każdy interwał

Tabela `daily_decision_replay_15m` zawiera tylko decyzje zwycięskiej strategii.
Pozostałe scenariusze pozostają w backteście jako warianty porównawcze.

Znaczenie głównych pól:

- `system_recommendation`: wykonalne `CHARGE`, `DISCHARGE` albo `HOLD`,
- `recommended_battery_energy_mwh`: dodatnia wartość oznacza ładowanie,
  ujemna rozładowanie,
- `decision_reason`: przyczyna decyzji,
- `energy_action`: rzeczywisty kierunek przepływu,
- `interval_gain_vs_no_battery_usd`: wartość decyzji względem scenariusza bazowego,
- `cumulative_day_gain_vs_no_battery_usd`: skumulowany efekt w danym dniu.

In [0]:
counterfactual_revenue_df = (
    backtest_df
    .groupBy("decision_id")
    .pivot("scenario", EXPECTED_SCENARIOS)
    .agg(F.first("net_revenue_usd"))
    .withColumnRenamed(
        "NO_BATTERY",
        "no_battery_interval_revenue_usd",
    )
    .withColumnRenamed(
        "BATTERY_BALANCING_PV_ONLY",
        "balancing_interval_revenue_usd",
    )
    .withColumnRenamed(
        "BATTERY_PRICE_GRID",
        "price_grid_interval_revenue_usd",
    )
)

selected_df = (
    backtest_df
    .filter(F.col("scenario") == RECOMMENDED_STRATEGY)
)

reason_labels = F.create_map(
    F.lit("BASELINE_DIRECT_PV_EXPORT"),
    F.lit("Bezpośredni eksport PV — brak magazynu"),
    F.lit("PREDICTED_PV_SURPLUS"),
    F.lit("Prognozowana nadwyżka PV względem harmonogramu"),
    F.lit("PREDICTED_PV_DEFICIT"),
    F.lit("Prognozowany niedobór PV względem harmonogramu"),
    F.lit("LOW_PREDICTED_PRICE"),
    F.lit("Niska prognozowana cena"),
    F.lit("HIGH_PREDICTED_PRICE"),
    F.lit("Wysoka prognozowana cena"),
    F.lit("MID_PRICE_PV_SURPLUS"),
    F.lit("Średnia cena i nadwyżka PV"),
    F.lit("MID_PRICE_PV_DEFICIT"),
    F.lit("Średnia cena i niedobór PV"),
    F.lit("NO_ACTION"),
    F.lit("Brak potrzeby działania"),
)

daily_window = (
    Window
    .partitionBy("delivery_date_local")
    .orderBy("target_interval_start_utc")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

replay_df = (
    selected_df
    .join(counterfactual_revenue_df, on="decision_id", how="left")
    .withColumn(
        "target_time_local",
        F.from_utc_timestamp(
            "target_interval_start_utc",
            "America/Los_Angeles",
        ),
    )
    .withColumn(
        "delivery_date_local",
        F.to_date("target_time_local"),
    )
    .withColumnRenamed("scenario", "recommended_strategy")
    .withColumnRenamed(
        "recommended_action",
        "raw_strategy_recommendation",
    )
    .withColumnRenamed("action", "system_recommendation")
    .withColumn(
        "recommended_battery_energy_mwh",
        F.when(
            F.col("system_recommendation") == "CHARGE",
            F.col("battery_charge_mwh"),
        )
        .when(
            F.col("system_recommendation") == "DISCHARGE",
            -F.col("battery_discharge_mwh"),
        )
        .otherwise(F.lit(0.0)),
    )
    .withColumn(
        "recommended_battery_power_mw",
        F.col("recommended_battery_energy_mwh") / F.lit(0.25),
    )
    .withColumn(
        "decision_reason_pl",
        F.coalesce(
            reason_labels[F.col("decision_reason")],
            F.col("decision_reason"),
        ),
    )
    .withColumn(
        "constraint_status",
        F.when(
            (
                F.col("raw_strategy_recommendation").isin(
                    "CHARGE", "DISCHARGE"
                )
            )
            & (F.col("system_recommendation") == "HOLD"),
            F.lit("ACTION_BLOCKED_BY_TECHNICAL_LIMIT"),
        ).otherwise(F.lit("ACTION_WITHIN_TECHNICAL_LIMITS")),
    )
    .withColumn(
        "operator_instruction",
        F.when(
            F.col("system_recommendation") == "CHARGE",
            F.format_string(
                "CHARGE %.4f MWh FROM %s",
                F.col("battery_charge_mwh"),
                F.col("charge_source"),
            ),
        )
        .when(
            F.col("system_recommendation") == "DISCHARGE",
            F.format_string(
                "DISCHARGE %.4f MWh TO GRID",
                F.col("battery_discharge_mwh"),
            ),
        )
        .otherwise(F.lit("HOLD — NO BATTERY TRANSFER")),
    )
    .withColumn(
        "interval_gain_vs_no_battery_usd",
        F.col("net_revenue_usd")
        - F.col("no_battery_interval_revenue_usd"),
    )
    .withColumn(
        "cumulative_day_net_revenue_usd",
        F.sum("net_revenue_usd").over(daily_window),
    )
    .withColumn(
        "cumulative_day_gain_vs_no_battery_usd",
        F.sum("interval_gain_vs_no_battery_usd").over(daily_window),
    )
    .withColumn(
        "recommendation_scope",
        F.lit("HISTORICAL_BACKTEST_SELECTED_STRATEGY"),
    )
    .withColumn("_created_at_utc", F.current_timestamp())
)

ordered_columns = [
    "decision_id",
    "decision_time_utc",
    "target_interval_start_utc",
    "target_interval_end_utc",
    "target_time_local",
    "delivery_date_local",
    "recommended_strategy",
    "system_recommendation",
    "operator_instruction",
    "decision_reason",
    "decision_reason_pl",
    "price_signal",
    "prediction_source",
    "constraint_status",
    "energy_action",
    "charge_source",
    "recommended_battery_energy_mwh",
    "recommended_battery_power_mw",
    "dam_schedule_mwh",
    "predicted_pv_energy_mwh",
    "actual_pv_energy_mwh",
    "predicted_price_usd_mwh",
    "actual_fmm_price_usd_mwh",
    "dam_price_usd_mwh",
    "soc_before_mwh",
    "soc_after_mwh",
    "pv_to_grid_mwh",
    "pv_to_battery_mwh",
    "grid_to_battery_mwh",
    "battery_to_grid_mwh",
    "net_export_mwh",
    "schedule_deviation_mwh",
    "curtailed_energy_mwh",
    "dam_revenue_usd",
    "realtime_adjustment_usd",
    "degradation_cost_usd",
    "net_revenue_usd",
    "no_battery_interval_revenue_usd",
    "balancing_interval_revenue_usd",
    "price_grid_interval_revenue_usd",
    "interval_gain_vs_no_battery_usd",
    "cumulative_day_net_revenue_usd",
    "cumulative_day_gain_vs_no_battery_usd",
    "recommendation_scope",
    "_created_at_utc",
]

replay_output_df = replay_df.select(*ordered_columns)

(
    replay_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REPLAY_TABLE)
)

print(f"Zapisano: {REPLAY_TABLE}")
display(
    replay_output_df
    .select(
        "target_time_local",
        "system_recommendation",
        "operator_instruction",
        "decision_reason_pl",
        "predicted_pv_energy_mwh",
        "dam_schedule_mwh",
        "soc_before_mwh",
        "soc_after_mwh",
        "interval_gain_vs_no_battery_usd",
        "cumulative_day_gain_vs_no_battery_usd",
    )
    .orderBy("target_interval_start_utc")
)


## 4. Testy końcowe systemu decyzyjnego

Notebook kończy się błędem, jeśli:

- nie ma dokładnie trzech scenariuszy,
- ranking nie wskazuje jednego zwycięzcy,
- replay nie ma jednej decyzji na interwał,
- brakuje rekomendacji lub wariantów porównawczych,
- przychód replay nie zgadza się z podsumowaniem zwycięskiej strategii.

In [0]:
saved_ranking_df = spark.table(RANKING_TABLE)
saved_kpi_df = spark.table(KPI_TABLE)
saved_replay_df = spark.table(REPLAY_TABLE)

source_scenarios = {
    row["scenario"]
    for row in backtest_df.select("scenario").distinct().collect()
}

selected_source_count = (
    backtest_df
    .filter(F.col("scenario") == RECOMMENDED_STRATEGY)
    .count()
)

duplicate_count = (
    saved_replay_df
    .groupBy("decision_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_recommendation_count = (
    saved_replay_df
    .filter(
        F.col("system_recommendation").isNull()
        | F.col("operator_instruction").isNull()
        | F.col("decision_reason").isNull()
    )
    .count()
)

missing_counterfactual_count = (
    saved_replay_df
    .filter(
        F.col("no_battery_interval_revenue_usd").isNull()
        | F.col("balancing_interval_revenue_usd").isNull()
        | F.col("price_grid_interval_revenue_usd").isNull()
    )
    .count()
)

replay_revenue = saved_replay_df.agg(
    F.sum("net_revenue_usd").alias("value")
).first()["value"]

expected_raw_revenue = float(winner["raw_net_revenue_usd"])

test_results = {
    "scenario_set_is_complete":
        source_scenarios == set(EXPECTED_SCENARIOS),
    "exactly_one_recommended_strategy":
        saved_ranking_df
        .filter(F.col("is_recommended_strategy"))
        .count() == 1,
    "exactly_one_kpi_row": saved_kpi_df.count() == 1,
    "replay_row_count_matches_selected_strategy":
        saved_replay_df.count() == selected_source_count,
    "duplicate_decision_count_is_zero": duplicate_count == 0,
    "missing_recommendation_count_is_zero":
        missing_recommendation_count == 0,
    "missing_counterfactual_count_is_zero":
        missing_counterfactual_count == 0,
    "replay_revenue_matches_summary":
        abs(float(replay_revenue) - expected_raw_revenue) <= 1e-6,
}

for test_name, passed in test_results.items():
    print(f"{test_name}: {'PASS' if passed else 'FAIL'}")

failed_tests = [
    name for name, passed in test_results.items() if not passed
]

if failed_tests:
    raise RuntimeError(
        "Testy systemu decyzyjnego nie zostały zaliczone: "
        + ", ".join(failed_tests)
    )

print()
print("Wszystkie testy systemu decyzyjnego zostały zaliczone.")
print(f"Rekomendowana strategia: {RECOMMENDED_STRATEGY}")
print(f"Liczba decyzji operatora: {saved_replay_df.count()}")
print(f"Tabela rankingu: {RANKING_TABLE}")
print(f"Tabela KPI: {KPI_TABLE}")
print(f"Tabela replay: {REPLAY_TABLE}")
